# NFL Model Artifact & Feature Analysis Notebook

This template walks through loading trained artifacts, inspecting engineered features, and performing exploratory & modeling analysis. Follow cells in order; adapt as needed.


In [ ]:
# Section 1: Set Up Environment
import os, json, math, textwrap, pickle, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)
random.seed(42)
np.random.seed(42)

ARTIFACTS_DIR = Path('../artifacts').resolve()
RAW_DATA_FALLBACK = Path('../data/raw')  # adjust if needed
print('Artifacts dir:', ARTIFACTS_DIR)
list(ARTIFACTS_DIR.glob('*.pkl'))[:5]

In [ ]:
# Section 2: Load or Create Input Data (Model Artifacts + Feature Frame)
from datetime import datetime
from collections import defaultdict

artifact_files = sorted(ARTIFACTS_DIR.glob('*_rf_v2_*.pkl'))
print(f'Found {len(artifact_files)} artifacts')
artifact_files

In [ ]:
# Helper: choose most recent artifact per target
import re
latest_by_target = {}
pattern = re.compile(r'(spread|total_points|binary_spread_label|binary_ou_label)_rf_v2_(\d{8}_\d{6})\.pkl')
for f in artifact_files:
    m = pattern.match(f.name)
    if not m: continue
    tgt, ts = m.groups()
    if tgt not in latest_by_target or ts > latest_by_target[tgt][1]:
        latest_by_target[tgt] = (f, ts)
latest_by_target

In [ ]:
# Load one artifact (change key as needed)
sel_target = 'spread'
art_path, ts = latest_by_target[sel_target]
with open(art_path, 'rb') as f:
    art = pickle.load(f)
print(sel_target, art.metrics, 'feature_count:', len(art.feature_columns))
art.feature_columns[:15]

In [ ]:
# Rebuild feature dataset (ensures code & artifact alignment)
from src.models.xgboost_randomforrest_model_v2 import NFLModelV2
model = NFLModelV2(target=sel_target)
model.load_games(start_season=2021)
model.build_feature_matrices(include_defense=True, include_differentials=True)
model.build_dataset()
full_df = model._dataset.copy()
full_df.head()

In [ ]:
# Section 3: Inspect Raw Data Structure
full_df.info(memory_usage='deep')
print('Shape:', full_df.shape)
full_df.describe().T.head(20)

In [ ]:
# Section 4: Basic Data Cleaning (illustrative - adapt as needed)
clean_df = full_df.copy()
# Example: fill any remaining numeric NaNs with column medians
num_cols = clean_df.select_dtypes(include=[np.number]).columns
clean_df[num_cols] = clean_df[num_cols].apply(lambda s: s.fillna(s.median()))
# Drop duplicate game rows if any (should not happen)
clean_df = clean_df.drop_duplicates(subset=['gamesummaryid']).reset_index(drop=True)
clean_df.head(3)

In [ ]:
# Section 5: Exploratory Data Analysis (EDA)
fig, axes = plt.subplots(1,3, figsize=(14,4))
clean_df['spread'].hist(ax=axes[0]); axes[0].set_title('Spread Dist')
clean_df['total_points'].hist(ax=axes[1]); axes[1].set_title('Total Points Dist')
clean_df['binary_spread_label'].value_counts().plot(kind='bar', ax=axes[2]); axes[2].set_title('Binary Spread Label')
plt.tight_layout()
plt.show()

In [ ]:
# Section 6: Statistical Summaries
summaries = {
    'points_mean_by_season': clean_df.groupby('season')['total_points'].mean(),
    'spread_abs_mean_by_season': clean_df.assign(abs_spread=clean_df['spread'].abs()).groupby('season')['abs_spread'].mean(),
}
summaries

In [ ]:
# Section 7: Data Transformation & Feature Engineering
fe_df = clean_df.copy()
fe_df['home_favorite'] = (fe_df['spread'] < 0).astype(int)
fe_df['points_diff'] = fe_df['homescore'] - fe_df['awayscore']
fe_df[['home_favorite','points_diff']].head()

In [ ]:
# Section 8: Visualization Dashboards
sns.set_theme(context='notebook', style='whitegrid')
plt.figure(figsize=(10,5))
season_pts = fe_df.groupby('season')['total_points'].mean()
season_pts.plot(marker='o')
plt.title('Average Total Points by Season')
plt.ylabel('Mean Total Points')
plt.show()

In [ ]:
# Section 9: Correlation & Dependency Analysis
corr_cols = [c for c in fe_df.columns if c.endswith('__off_hist')][:25]
subset = fe_df[corr_cols].dropna(axis=1, how='all')
plt.figure(figsize=(14,10))
sns.heatmap(subset.corr(), cmap='coolwarm', center=0)
plt.title('Correlation (subset of off_hist features)')
plt.show()

In [ ]:
# Section 10: Simple Predictive Modeling (Optional)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

model_cols = [c for c in fe_df.columns if c.endswith('__off_hist')][:40]
X = fe_df[model_cols].fillna(0)
y = fe_df['total_points']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
rf_tmp = RandomForestRegressor(n_estimators=200, random_state=42)
rf_tmp.fit(X_train, y_train)
from sklearn.metrics import mean_absolute_error
preds = rf_tmp.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print('Temp RF MAE:', round(mae,3))

In [ ]:
# Section 11: Model Evaluation & Metrics
residuals = y_test - preds
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(residuals, bins=20)
plt.title('Residual Distribution')
plt.subplot(1,2,2)
plt.scatter(preds, residuals, alpha=0.6)
plt.axhline(0, color='r')
plt.title('Residuals vs Predictions')
plt.tight_layout()
plt.show()

In [ ]:
# Section 12: Iterative Experiment Tracking
from datetime import datetime
exp_log_path = Path('../artifacts/experiment_log.json')
entry = {'timestamp': datetime.utcnow().isoformat(), 'temp_rf_mae': float(mae), 'features': model_cols}
if exp_log_path.exists():
    existing = json.loads(exp_log_path.read_text())
    if isinstance(existing, list):
        existing.append(entry)
    else:
        existing = [existing, entry]
else:
    existing = [entry]
exp_log_path.write_text(json.dumps(existing, indent=2))
existing[-3:]

In [ ]:
# Section 13: Export Processed Data & Artifacts
export_dir = Path('../artifacts/notebook_exports')
export_dir.mkdir(exist_ok=True)
clean_df.to_parquet(export_dir / 'clean_games.parquet', index=False)
print('Saved clean games to', export_dir)

In [ ]:
# Section 14: Create Reproducible Pipeline Functions
def build_dataset(target: str, start_season: int = 2021):
    m = NFLModelV2(target=target)
    m.load_games(start_season)
    m.build_feature_matrices(include_defense=True, include_differentials=True)
    m.build_dataset()
    return m

def train_baseline(target: str):
    m = build_dataset(target)
    metrics = m.fit_random_forest()
    return m, metrics

# Quick smoke test
_, smoke_metrics = train_baseline('binary_spread_label')
smoke_metrics

In [ ]:
# Section 15: Automate Notebook Cells for Reruns

def main():
    targets = ['spread','total_points','binary_spread_label','binary_ou_label']
    results = {}
    for t in targets:
        m, metrics = train_baseline(t)
        results[t] = metrics
    return results

# Uncomment to run full pipeline (can take time)
# all_results = main()
# all_results